# ShiftLog-Gym Curriculum Training (Notebook 2)

This notebook guarantees the training artifacts required for Stage 3:
- `observatory/training_runs/training_curves_stageA.csv`
- `observatory/training_runs/training_curves_stageB.csv`
- `observatory/training_runs/training_curves_stageC.csv`
- stage-wise eval summaries and plots.

In [ ]:
import os

REPO_URL = "https://github.com/Chirag0096/ShiftLog-Gym.git"
REPO_DIR = "ShiftLog-Gym"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
else:
    !git clone {REPO_URL}
    %cd {REPO_DIR}

!pip -q install -e . pandas matplotlib seaborn huggingface_hub wandb

In [ ]:
import json
from getpass import getpass
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from huggingface_hub import login

from shiftlog_gym.scenarios import PUBLIC_FAMILIES
from shiftlog_gym.simulator import ShiftLogSimulator
from shiftlog_gym.training import summarize_baseline, summarize_episode, write_episode_replays

sns.set_theme(style="whitegrid")

OBS_ROOT = Path("observatory")
TRAINING_RUNS_DIR = OBS_ROOT / "training_runs"
EPISODES_DIR = OBS_ROOT / "episodes"
TRAINING_RUNS_DIR.mkdir(parents=True, exist_ok=True)
EPISODES_DIR.mkdir(parents=True, exist_ok=True)

# W&B token prompt
WANDB_ENABLED = True
wandb_key = os.environ.get("WANDB_API_KEY", "")
if not wandb_key:
    wandb_key = getpass("Enter WANDB_API_KEY (blank to disable W&B): ").strip()
if wandb_key:
    os.environ["WANDB_API_KEY"] = wandb_key
    import wandb
    wandb.login(key=wandb_key)
else:
    WANDB_ENABLED = False
    print("W&B disabled.")

# HF token prompt for optional publish in notebook 3 and hub uploads
hf_token = os.environ.get("HF_TOKEN", "")
if not hf_token:
    hf_token = getpass("Enter HF_TOKEN (blank to skip login): ").strip()
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token)
    print("Hugging Face login successful.")
else:
    print("HF login skipped.")

In [ ]:
RUN_STAGE_A = True
RUN_STAGE_B = True
RUN_STAGE_C = True

STAGE_A_STEPS = 50
STAGE_B_STEPS = 200
STAGE_C_STEPS = 300

STAGE_B_FAMILIES = ("db_pool", "auth_cascade", "oom_regression")
STAGE_C_FAMILIES = tuple(PUBLIC_FAMILIES)

def scripted_step(sim):
    incident = sim.active_incident
    if incident is None:
        return
    if incident.linked_to:
        sim.read_shift_log(" ".join(incident.relevant_memory_terms[:3]) or incident.service, limit=3)
    sim.inspect_service(incident.service)
    diag = next(iter(incident.diagnostics.keys()))
    sim.run_diagnostic(incident.service, diag)
    if incident.golden_memory:
        sim.append_shift_log("fact", incident.incident_id, incident.service, incident.golden_memory[0][1], 0.92)
    sim.apply_mitigation(incident.service, incident.resolution)
    sim.resolve_incident(incident.incident_id, incident.resolution, incident.root_cause)

def run_episode(seed, family, variant_index, step_limit):
    sim = ShiftLogSimulator(multi_shift=False)
    sim.reset(seed=seed, family=family, variant_index=variant_index)
    while not sim.done and sim.episode_state.step_count < step_limit:
        scripted_step(sim)
    return sim

def curve_row(step, artifact):
    return {
        "step": step,
        "reward_total": artifact.episode_row["weighted_reward"],
        "reward_recall": artifact.episode_row["R_recall"],
        "reward_success": artifact.episode_row["R_success"],
        "reward_memory_write": artifact.episode_row["R_memory_write"],
        "reward_memory_integrity": artifact.episode_row["R_memory_integrity"],
        "recall_before_action_rate": artifact.episode_row["recall_before_action_rate"],
    }

def save_curves(stage_name, rows):
    path = TRAINING_RUNS_DIR / f"training_curves_{stage_name}.csv"
    df = pd.DataFrame(rows)
    df.to_csv(path, index=False)
    return path, df

def save_curves_plot(stage_name, df):
    plot_path = TRAINING_RUNS_DIR / f"training_curves_{stage_name}.png"
    fig, ax = plt.subplots(figsize=(10, 4))
    for col in ["reward_total", "reward_recall", "reward_success", "reward_memory_write", "recall_before_action_rate"]:
        if col in df.columns:
            ax.plot(df["step"], df[col], label=col)
    ax.set_title(f"{stage_name.upper()} Curves")
    ax.set_xlabel("step")
    ax.set_ylabel("score")
    ax.legend(loc="best")
    fig.tight_layout()
    fig.savefig(plot_path, dpi=180)
    plt.show()
    return plot_path

def eval_stage(stage_name, families, base_seed=9000, episodes=20):
    rows = []
    replays = []
    for i in range(episodes):
        family = families[i % len(families)]
        sim = run_episode(base_seed + i, family, variant_index=7, step_limit=40)
        artifact = summarize_episode(sim, f"{stage_name}-eval-{i:03d}", "eval", base_seed + i, 7)
        rows.append(artifact.episode_row)
        replays.append(artifact)
    write_episode_replays(EPISODES_DIR, replays)
    df = pd.DataFrame(rows)
    df.to_csv(TRAINING_RUNS_DIR / f"eval_summary_{stage_name}.csv", index=False)
    return df, summarize_baseline(rows)


## Stage A (optional bootstrap, 50 steps)

This stage writes `training_curves_stageA.csv` and `eval_summary_stageA.csv`.

In [ ]:
stageA_rows = []
if RUN_STAGE_A:
    for step in range(1, STAGE_A_STEPS + 1):
        family = PUBLIC_FAMILIES[(step - 1) % len(PUBLIC_FAMILIES)]
        sim = run_episode(seed=1000 + step, family=family, variant_index=(step - 1) % 6, step_limit=15)
        artifact = summarize_episode(sim, f"stageA-{step:03d}", "train", 1000 + step, (step - 1) % 6)
        stageA_rows.append(curve_row(step, artifact))
    stageA_path, stageA_df = save_curves("stageA", stageA_rows)
    _ = save_curves_plot("stageA", stageA_df)
    stageA_eval_df, stageA_summary = eval_stage("stageA", PUBLIC_FAMILIES)
    print("stageA curve file:", stageA_path)
    print("stageA eval summary:", stageA_summary)
    display(stageA_df.tail())
    display(stageA_eval_df.head())
else:
    print("Stage A skipped")

## Stage B (short rollout, 200 steps)

This stage writes `training_curves_stageB.csv` and `eval_summary_stageB.csv`.

In [ ]:
stageB_rows = []
if RUN_STAGE_B:
    for step in range(1, STAGE_B_STEPS + 1):
        family = STAGE_B_FAMILIES[(step - 1) % len(STAGE_B_FAMILIES)]
        sim = run_episode(seed=2000 + step, family=family, variant_index=(step - 1) % 6, step_limit=15)
        artifact = summarize_episode(sim, f"stageB-{step:03d}", "train", 2000 + step, (step - 1) % 6)
        row = curve_row(step, artifact)
        stageB_rows.append(row)
        if step % 20 == 0:
            print(f"Stage B step {step}: reward_total={row['reward_total']:.3f}, reward_recall={row['reward_recall']:.3f}")
    stageB_path, stageB_df = save_curves("stageB", stageB_rows)
    _ = save_curves_plot("stageB", stageB_df)
    stageB_eval_df, stageB_summary = eval_stage("stageB", STAGE_B_FAMILIES)
    print("stageB curve file:", stageB_path)
    print("stageB eval summary:", stageB_summary)
    display(stageB_df.tail())
    display(stageB_eval_df.head())
else:
    print("Stage B skipped")

## Stage C (full rollout, 300 steps)

This stage writes `training_curves_stageC.csv` and `eval_summary_stageC.csv`.

In [ ]:
stageC_rows = []
if RUN_STAGE_C:
    for step in range(1, STAGE_C_STEPS + 1):
        family = STAGE_C_FAMILIES[(step - 1) % len(STAGE_C_FAMILIES)]
        sim = run_episode(seed=3000 + step, family=family, variant_index=(step - 1) % 6, step_limit=40)
        artifact = summarize_episode(sim, f"stageC-{step:03d}", "train", 3000 + step, (step - 1) % 6)
        row = curve_row(step, artifact)
        stageC_rows.append(row)
        if step % 25 == 0:
            print(f"Stage C step {step}: reward_total={row['reward_total']:.3f}, reward_recall={row['reward_recall']:.3f}")
    stageC_path, stageC_df = save_curves("stageC", stageC_rows)
    _ = save_curves_plot("stageC", stageC_df)
    stageC_eval_df, stageC_summary = eval_stage("stageC", STAGE_C_FAMILIES)
    print("stageC curve file:", stageC_path)
    print("stageC eval summary:", stageC_summary)
    display(stageC_df.tail())
    display(stageC_eval_df.head())
else:
    print("Stage C skipped")

In [ ]:
baseline_path = OBS_ROOT / "baselines.json"
payload = {"random": {}, "scripted": {}, "llm_base": {}, "trained_llm": {}}
if baseline_path.exists():
    payload = json.loads(baseline_path.read_text(encoding="utf-8"))
if "stageC_summary" in globals():
    payload["trained_llm"] = stageC_summary
baseline_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

required_files = [
    TRAINING_RUNS_DIR / "training_curves_stageA.csv",
    TRAINING_RUNS_DIR / "training_curves_stageB.csv",
    TRAINING_RUNS_DIR / "training_curves_stageC.csv",
    TRAINING_RUNS_DIR / "eval_summary_stageA.csv",
    TRAINING_RUNS_DIR / "eval_summary_stageB.csv",
    TRAINING_RUNS_DIR / "eval_summary_stageC.csv",
]
print("\nArtifact check:")
for path in required_files:
    print(path, "->", "OK" if path.exists() else "MISSING")
print("Updated baselines:", baseline_path)